# Inference Notebook - Testing All Recommendation Algorithms (COPILOT code)

This notebook tests all 7 recommendation algorithms with sample queries to evaluate their performance and behavior.

## 1. Setup and Load Data

In [1]:
import pandas as pd
import numpy as np
import pickle
import gc
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

data_path = '../../data/all_recipes_final.csv'
df = pd.read_csv(data_path)

print(f"Loaded {len(df)} recipes")
print(f"Columns: {df.columns.tolist()}")

Loaded 10263 recipes
Columns: ['title', 'type_of_food', 'link', 'description', 'ingredients', 'ingredients_normalized', 'step', 'note', 'num_of_ingredients', 'cook_time', 'num_of_people', 'calories', 'source']


## 2. Define Test Queries

We'll test 5 different types of queries to see how each algorithm performs:

In [2]:
test_queries = [
    "Thịt kho nước dừa",
    "Món ăn từ gà và nấm",
    "Canh chua cá lóc",
    "Bánh ngọt làm từ trứng",
    "Món chay từ đậu hũ"
]

print("Test Queries:")
for i, query in enumerate(test_queries, 1):
    print(f"  {i}. {query}")

Test Queries:
  1. Thịt kho nước dừa
  2. Món ăn từ gà và nấm
  3. Canh chua cá lóc
  4. Bánh ngọt làm từ trứng
  5. Món chay từ đậu hũ


## 3. Helper Function to Display Results

In [3]:
def display_recommendations(query, recommendations, top_k=5):
    """Display the top k recommendations for a given query"""
    print(f"\n{'='*80}")
    print(f"Query: '{query}'")
    print(f"{'='*80}")
    
    if recommendations is None or len(recommendations) == 0:
        print("No recommendations found")
        return
    
    for i, (idx, score) in enumerate(recommendations[:top_k], 1):
        recipe = df.iloc[idx]
        print(f"\n{i}. {recipe['title']}")
        print(f"   Score: {score:.4f}")
        print(f"   Type: {recipe['type_of_food']}")
        print(f"   Ingredients: {recipe['ingredients'][:100]}...")
        print(f"   Source: {recipe['source']}")

---
# Algorithm Testing Sections

## Section 1: TF-IDF Based Recommendation

TF-IDF (Term Frequency-Inverse Document Frequency) vectorization with cosine similarity.

In [4]:
with open('../Saved_models/TFIDF/tfidf_vectorizer.pkl', 'rb') as f:
    tfidf_vectorizer = pickle.load(f)

with open('../Saved_models/TFIDF/tfidf_processed_data.pkl', 'rb') as f:
    tfidf_processed_data = pickle.load(f)

tfidf_similarity = np.load('../Saved_models/TFIDF/tfidf_similarity.npy')

def get_recommendations_from_similarity(query_idx, similarity_matrix, top_k=10):
    """Get top k recommendations from precomputed similarity matrix"""
    scores = similarity_matrix[query_idx].copy()
    scores[query_idx] = -1
    top_indices = np.argsort(scores)[::-1][:top_k]
    recommendations = [(idx, scores[idx]) for idx in top_indices]
    return recommendations

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"Query: '{query}'")
    print(f"{'='*80}")
    
    query_lower = query.lower()
    best_match_idx = None
    best_match_score = 0
    
    for idx, row in df.iterrows():
        score = 0
        title_lower = str(row['title']).lower()
        for word in query_lower.split():
            if len(word) > 2 and word in title_lower:
                score += 1
        if score > best_match_score:
            best_match_score = score
            best_match_idx = idx
    
    if best_match_idx is not None:
        print(f"Using recipe: {df.iloc[best_match_idx]['title']}")
        recommendations = get_recommendations_from_similarity(best_match_idx, tfidf_similarity)
        display_recommendations(query, recommendations)
    else:
        print("No matching recipe found")

del tfidf_vectorizer, tfidf_processed_data, tfidf_similarity
gc.collect()


Query: 'Thịt kho nước dừa'
Using recipe: Vịt kho nước dừa tươi thơm mềm, ngọt thịt tại nhà cực đơn giản

Query: 'Thịt kho nước dừa'

1. Vịt khìa nước dừa đậm đà, ngon miệng, thơm lừng nức mũi
   Score: 0.7315
   Type: Món chiên
   Ingredients: ['10 g Hành tím', '10 g Tỏi', '1 trái Dừa', '1 con Vịt', '1 ít Gia vị thông dụng (bột ngũ vị hương/b...
   Source: dienmayxanh

2. Món vịt chiên nước dừa hương vị đậm đà, khó quên
   Score: 0.7215
   Type: Món chiên
   Ingredients: ['1.5 kg Vịt nguyên con', '1 củ Tỏi', '5 củ Hành tím', '1.5 trái Nước dừa tươi', '3 quả Ớt', '3 muỗn...
   Source: dienmayxanh

3. Vịt om sấu nước dừa thơm ngọt, chua thanh, đơn giản tại nhà
   Score: 0.7170
   Type: Món nước
   Ingredients: ['1 kg Thịt vịt', '10 quả Sấu', '500 ml Nước dừa tươi', '1 củ Gừng', '1 củ Riềng', '3 nhánh Hành lá'...
   Source: dienmayxanh

4. Thịt vịt kho sả ớt cay nồng, bắt vị cho bữa cơm hấp dẫn
   Score: 0.7065
   Type: Món kho
   Ingredients: ['1/2 con Vịt ta (hoặc vịt xiêm)', '4 trái Ớ

11

---
## Section 2: Keyword Based Recommendation

Simple keyword matching based on ingredient overlap.

In [5]:
keyword_similarity = np.load('../Saved_models/Keyword/keyword_similarity.npy')

def keyword_recommend(query, df, top_k=10):
    """Simple keyword matching"""
    query_lower = query.lower()
    scores = []
    
    for idx, row in df.iterrows():
        score = 0
        title_lower = str(row['title']).lower()
        ingredients_lower = str(row['ingredients']).lower()
        
        for word in query_lower.split():
            if len(word) > 2:
                if word in title_lower:
                    score += 2
                if word in ingredients_lower:
                    score += 1
        
        scores.append(score)
    
    top_indices = np.argsort(scores)[-top_k:][::-1]
    recommendations = [(idx, scores[idx]) for idx in top_indices if scores[idx] > 0]
    
    return recommendations

for query in test_queries:
    recommendations = keyword_recommend(query, df)
    display_recommendations(query, recommendations)

del keyword_similarity
gc.collect()


Query: 'Thịt kho nước dừa'

1. Vịt kho nước dừa tươi thơm mềm, ngọt thịt tại nhà cực đơn giản
   Score: 11.0000
   Type: Món kho
   Ingredients: ['1 kg Thịt vịt', '700 ml Nước dừa', '100 gr Gừng', '1 ít Ớt', '1.5 muỗng canh Tỏi băm', '1 muỗng ca...
   Source: dienmayxanh

2. Thịt kho trứng cút nước dừa thơm ngon bắt cơm
   Score: 11.0000
   Type: Món kho
   Ingredients: ['300 gr Thịt ba chỉ', '15 quả Trứng cút', '1 quả Dừa xiêm', '3 nhánh Hành lá', '1 củ Gừng', '2 quả ...
   Source: dienmayxanh

3. Thịt kho trứng không cần nước dừa vẫn ngon đậm đà cực hao cơm
   Score: 10.0000
   Type: Món kho
   Ingredients: ['500 gr Thịt heo (nên dùng thịt ba chỉ sẽ ngon hơn)', '10 trái Trứng gà (có thể thay bằng trứng vịt...
   Source: dienmayxanh

4. Công thức món thịt kho trứng, cùi dừa THƠM NGON, ĐẬM ĐÀ
   Score: 10.0000
   Type: Món ngon hàng ngày
   Ingredients: ['a) Ướp thịt lợn:', '1 kg thịt lợn ba chỉ. Nên chọn thịt mỡ trắng, thớ thịt tươi hồng, săn chắc.', ...
   Source: vnexpress

5. Cá n

11

---
## Section 3: Ingredient TF-IDF Based Recommendation

TF-IDF vectorization focused specifically on ingredients.

In [6]:
import os
ingredient_path = '../Saved_models/Ingredient_TFIDF/'

with open(ingredient_path + 'ingredient_tfidf_vectorizer.pkl', 'rb') as f:
    ingredient_tfidf_vectorizer = pickle.load(f)

ingredient_tfidf_similarity = np.load(ingredient_path + 'ingredient_tfidf_similarity.npy')

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"Query: '{query}'")
    print(f"{'='*80}")
    
    query_lower = query.lower()
    best_match_idx = None
    best_match_score = 0
    
    for idx, row in df.iterrows():
        score = 0
        ingredients_lower = str(row['ingredients']).lower()
        for word in query_lower.split():
            if len(word) > 2 and word in ingredients_lower:
                score += 1
        if score > best_match_score:
            best_match_score = score
            best_match_idx = idx
    
    if best_match_idx is not None:
        print(f"Using recipe: {df.iloc[best_match_idx]['title']}")
        recommendations = get_recommendations_from_similarity(best_match_idx, ingredient_tfidf_similarity)
        display_recommendations(query, recommendations)
    else:
        print("No matching recipe found")

del ingredient_tfidf_vectorizer, ingredient_tfidf_similarity
gc.collect()


Query: 'Thịt kho nước dừa'
Using recipe: Cà ri bò bổ dưỡng cho ngày nồm ẩm

Query: 'Thịt kho nước dừa'

1. Món cà ri gà - món ấm áp cho ngày lạnh
   Score: 0.5031
   Type: Món ngon ngày lạnh
   Ingredients: ['1/2 con gà (khoảng 800 gr)', '2 củ khoai tây', '1 củ cà rốt', '1 củ hành tây', '1 hộp nước cốt dừa...
   Source: vnexpress

2. Bò sốt vang khoai tây cà rốt đơn giản tại nhà
   Score: 0.3953
   Type: Món từ bò
   Ingredients: ['500 gr Thịt nạm bò', '130 ml Rượu vang đỏ', '5 trái Cà chua', '2 củ Cà rốt', '2 củ Khoai tây', '1 ...
   Source: dienmayxanh

3. Bò hầm cà phê mới lạ thơm béo đơn giản ngon khó cưỡng
   Score: 0.3744
   Type: Món từ bò
   Ingredients: ['1 kg Sườn non bò', '1 củ Cà rốt', '1 củ Hành tây', '100 gr Cần tây', '3 nhánh Lá cần tàu', '1 nhán...
   Source: dienmayxanh

4. Bò sốt vang kiểu Pháp thơm ngon chuẩn vị
   Score: 0.3727
   Type: Món từ bò
   Ingredients: ['1 kg Thịt bò nạm (loại nhiều gân)', '3 lá Nguyệt quế', '1 hộp Sốt cà chua (khoảng 227gr)', '200 ml...


11

---
## Section 4: SBERT + FAISS Based Recommendation

Sentence-BERT embeddings with FAISS for efficient similarity search.

In [7]:
try:
    from sentence_transformers import SentenceTransformer
    import faiss
    import json
    
    sbert_dir = '../Saved_models/SBERT_FAISS/'
    with open(sbert_dir + 'model_info.json', 'r', encoding='utf-8') as f:
        sbert_info = json.load(f)
    
    sbert_model = SentenceTransformer(sbert_info['model_name'])
    sbert_embeddings = np.load(sbert_dir + 'recipe_embeddings.npy')
    
    from sklearn.metrics.pairwise import cosine_similarity
    sbert_similarity = cosine_similarity(sbert_embeddings, sbert_embeddings)
    
    for query in test_queries:
        print(f"\n{'='*80}")
        print(f"Query: '{query}'")
        print(f"{'='*80}")
        
        query_lower = query.lower()
        best_match_idx = None
        best_match_score = 0
        
        for idx, row in df.iterrows():
            score = 0
            title_lower = str(row['title']).lower()
            for word in query_lower.split():
                if len(word) > 2 and word in title_lower:
                    score += 1
            if score > best_match_score:
                best_match_score = score
                best_match_idx = idx
        
        if best_match_idx is not None:
            print(f"Using recipe: {df.iloc[best_match_idx]['title']}")
            recommendations = get_recommendations_from_similarity(best_match_idx, sbert_similarity)
            display_recommendations(query, recommendations)
        else:
            print("No matching recipe found")
    
    del sbert_model, sbert_embeddings, sbert_similarity
    gc.collect()
    
except Exception as e:
    print(f"Error loading SBERT + FAISS model: {e}")
    import traceback
    traceback.print_exc()


Query: 'Thịt kho nước dừa'
Using recipe: Vịt kho nước dừa tươi thơm mềm, ngọt thịt tại nhà cực đơn giản

Query: 'Thịt kho nước dừa'

1. Vịt rô ti với nước dừa mềm ngon, ngọt thịt bằng chảo sâu lòng
   Score: 0.9160
   Type: Món kho
   Ingredients: ['1 con Vịt (Khoảng 1.3kg)', '3 củ Hành tím', '1/2 củ Tỏi', '1 lít Nước dừa', '1 muỗng cà phê Bột ng...
   Source: dienmayxanh

2. Thịt vịt kho sả ớt cay nồng, bắt vị cho bữa cơm hấp dẫn
   Score: 0.9104
   Type: Món kho
   Ingredients: ['1/2 con Vịt ta (hoặc vịt xiêm)', '4 trái Ớt', '3 cây Sả tươi', '2 củ Gừng', '2 trái Chanh', '7 tép...
   Source: dienmayxanh

3. Chi tiết cách làm thịt kho Tàu thơm ngon đúng vị, thịt mềm béo ngậy
   Score: 0.9047
   Type: Món kho
   Ingredients: ['500 gr Thịt ba chỉ hay thịt chân giò', '5 quả Trứng vịt luộc', '400 ml Nước dừa', '1 muỗng canh Hà...
   Source: dienmayxanh

4. Thịt kho Tàu miền Nam ngon ngọt thơm lừng, mềm ngon đậm vị
   Score: 0.9023
   Type: Món kho
   Ingredients: ['500 gr Thịt ba chỉ', '4

---
## Section 5: Hybrid (TF-IDF + SBERT) Recommendation

Combines both TF-IDF and SBERT approaches for better results.

In [8]:
try:
    from sentence_transformers import SentenceTransformer
    import json
    
    hybrid_dir = '../Saved_models/Hybrid_TFIDF_SBERT/'
    with open(hybrid_dir + 'config.json', 'r', encoding='utf-8') as f:
        hybrid_config = json.load(f)
    
    alpha = hybrid_config['alpha']
    
    tfidf_sim_normalized = np.load('../Saved_models/TFIDF/tfidf_similarity.npy')
    hybrid_sbert_embeddings = np.load(hybrid_dir + 'sbert_embeddings.npy')
    
    sbert_sim = cosine_similarity(hybrid_sbert_embeddings, hybrid_sbert_embeddings)
    hybrid_tfidf_sbert_similarity = alpha * tfidf_sim_normalized + (1 - alpha) * sbert_sim
    
    for query in test_queries:
        print(f"\n{'='*80}")
        print(f"Query: '{query}'")
        print(f"{'='*80}")
        
        query_lower = query.lower()
        best_match_idx = None
        best_match_score = 0
        
        for idx, row in df.iterrows():
            score = 0
            title_lower = str(row['title']).lower()
            for word in query_lower.split():
                if len(word) > 2 and word in title_lower:
                    score += 1
            if score > best_match_score:
                best_match_score = score
                best_match_idx = idx
        
        if best_match_idx is not None:
            print(f"Using recipe: {df.iloc[best_match_idx]['title']}")
            recommendations = get_recommendations_from_similarity(best_match_idx, hybrid_tfidf_sbert_similarity)
            display_recommendations(query, recommendations)
        else:
            print("No matching recipe found")
    
    del tfidf_sim_normalized, hybrid_sbert_embeddings, sbert_sim, hybrid_tfidf_sbert_similarity
    gc.collect()
    
except Exception as e:
    print(f"Error loading Hybrid (TF-IDF + SBERT) model: {e}")
    import traceback
    traceback.print_exc()


Query: 'Thịt kho nước dừa'
Using recipe: Vịt kho nước dừa tươi thơm mềm, ngọt thịt tại nhà cực đơn giản

Query: 'Thịt kho nước dừa'

1. Món vịt chiên nước dừa hương vị đậm đà, khó quên
   Score: 0.8100
   Type: Món chiên
   Ingredients: ['1.5 kg Vịt nguyên con', '1 củ Tỏi', '5 củ Hành tím', '1.5 trái Nước dừa tươi', '3 quả Ớt', '3 muỗn...
   Source: dienmayxanh

2. Vịt khìa nước dừa đậm đà, ngon miệng, thơm lừng nức mũi
   Score: 0.8095
   Type: Món chiên
   Ingredients: ['10 g Hành tím', '10 g Tỏi', '1 trái Dừa', '1 con Vịt', '1 ít Gia vị thông dụng (bột ngũ vị hương/b...
   Source: dienmayxanh

3. Thịt vịt kho sả ớt cay nồng, bắt vị cho bữa cơm hấp dẫn
   Score: 0.8084
   Type: Món kho
   Ingredients: ['1/2 con Vịt ta (hoặc vịt xiêm)', '4 trái Ớt', '3 cây Sả tươi', '2 củ Gừng', '2 trái Chanh', '7 tép...
   Source: dienmayxanh

4. Vịt om sấu nước dừa thơm ngọt, chua thanh, đơn giản tại nhà
   Score: 0.8061
   Type: Món nước
   Ingredients: ['1 kg Thịt vịt', '10 quả Sấu', '500 ml Nước

---
## Section 6: Hybrid (General) Recommendation

Another hybrid approach combining multiple recommendation strategies.

In [9]:
try:
    hybrid_path = '../Saved_models/Hybrid/'
    hybrid_similarity = np.load(hybrid_path + 'hybrid_similarity.npy')
    
    for query in test_queries:
        print(f"\n{'='*80}")
        print(f"Query: '{query}'")
        print(f"{'='*80}")
        
        query_lower = query.lower()
        best_match_idx = None
        best_match_score = 0
        
        for idx, row in df.iterrows():
            score = 0
            title_lower = str(row['title']).lower()
            for word in query_lower.split():
                if len(word) > 2 and word in title_lower:
                    score += 1
            if score > best_match_score:
                best_match_score = score
                best_match_idx = idx
        
        if best_match_idx is not None:
            print(f"Using recipe: {df.iloc[best_match_idx]['title']}")
            recommendations = get_recommendations_from_similarity(best_match_idx, hybrid_similarity)
            display_recommendations(query, recommendations)
        else:
            print("No matching recipe found")
    
    del hybrid_similarity
    gc.collect()
        
except Exception as e:
    print(f"Error loading Hybrid (General) model: {e}")
    import traceback
    traceback.print_exc()


Query: 'Thịt kho nước dừa'
Using recipe: Vịt kho nước dừa tươi thơm mềm, ngọt thịt tại nhà cực đơn giản

Query: 'Thịt kho nước dừa'

1. Món vịt chiên nước dừa hương vị đậm đà, khó quên
   Score: 0.4960
   Type: Món chiên
   Ingredients: ['1.5 kg Vịt nguyên con', '1 củ Tỏi', '5 củ Hành tím', '1.5 trái Nước dừa tươi', '3 quả Ớt', '3 muỗn...
   Source: dienmayxanh

2. Vịt ram lá quế lạ miệng hấp dẫn đơn giản đổi vị cho bữa cơm
   Score: 0.4845
   Type: Món xào
   Ingredients: ['500 gr Ức vịt', '20 gr Lá húng quế', '10 gr Đậu phộng rang', '3 muỗng cà phê Hành tím băm', '3.5 m...
   Source: dienmayxanh

3. Hướng dẫn cách nấu thịt vịt giả cầy thơm ngon chuẩn vị người miền Bắc
   Score: 0.4607
   Type: Món xào
   Ingredients: ['1/2 con Vịt', '1 củ Riềng', '1 củ Tỏi', '1 củ Gừng', '3 thìa Mẻ', '10 gr Gia vị (bột nghệ/mắm tôm/...
   Source: dienmayxanh

4. Thịt vịt kho sả ớt cay nồng, bắt vị cho bữa cơm hấp dẫn
   Score: 0.4584
   Type: Món kho
   Ingredients: ['1/2 con Vịt ta (hoặc vịt xiêm)'

---
## Section 7: RA-Rec (Recipe Attribute Recommendation)

Advanced recommendation based on recipe attributes and user preferences.

In [10]:
try:
    from sentence_transformers import SentenceTransformer
    
    rarec_model = SentenceTransformer('keepitreal/vietnamese-sbert')
    
    rarec_path = '../RA_Rec/'
    embeddings_path = rarec_path + 'recipes_embeddings_list.pkl'
    
    with open(embeddings_path, 'rb') as f:
        recipes_embeddings_list = pickle.load(f)
    
    def late_fusion_inference(query_idx, model, recipes_embeddings_list, df, top_k=10):
        """Late Fusion: Average similarity across all sentences in each recipe"""
        query_recipe = df.iloc[query_idx]
        query_text = f"{query_recipe['title']}. {query_recipe['description']}"
        
        query_embedding = model.encode([query_text])
        query_embedding = query_embedding / np.linalg.norm(query_embedding)
        
        recipe_scores = []
        
        for recipe_idx, dish_embeds in enumerate(recipes_embeddings_list):
            if len(dish_embeds) == 0:
                continue
            
            if recipe_idx == query_idx:
                continue
            
            dish_embeds_norm = dish_embeds / np.linalg.norm(dish_embeds, axis=1, keepdims=True)
            similarities = np.dot(dish_embeds_norm, query_embedding.T).flatten()
            avg_similarity = np.mean(similarities)
            
            recipe_scores.append((recipe_idx, avg_similarity))
        
        recipe_scores.sort(key=lambda x: x[1], reverse=True)
        
        return recipe_scores[:top_k]
    
    for query in test_queries:
        print(f"\n{'='*80}")
        print(f"Query: '{query}'")
        print(f"{'='*80}")
        
        query_lower = query.lower()
        best_match_idx = None
        best_match_score = 0
        
        for idx, row in df.iterrows():
            score = 0
            title_lower = str(row['title']).lower()
            for word in query_lower.split():
                if len(word) > 2 and word in title_lower:
                    score += 1
            if score > best_match_score:
                best_match_score = score
                best_match_idx = idx
        
        if best_match_idx is not None:
            print(f"Using recipe: {df.iloc[best_match_idx]['title']}")
            recommendations = late_fusion_inference(best_match_idx, rarec_model, recipes_embeddings_list, df)
            display_recommendations(query, recommendations)
        else:
            print("No matching recipe found")
    
    del rarec_model, recipes_embeddings_list
    gc.collect()
    
except Exception as e:
    print(f"Error loading RA-Rec model: {e}")
    import traceback
    traceback.print_exc()


Query: 'Thịt kho nước dừa'
Using recipe: Vịt kho nước dừa tươi thơm mềm, ngọt thịt tại nhà cực đơn giản

Query: 'Thịt kho nước dừa'

1. Vịt khìa nước dừa đậm đà, ngon miệng, thơm lừng nức mũi
   Score: 0.6873
   Type: Món chiên
   Ingredients: ['10 g Hành tím', '10 g Tỏi', '1 trái Dừa', '1 con Vịt', '1 ít Gia vị thông dụng (bột ngũ vị hương/b...
   Source: dienmayxanh

2. Thịt heo quay kho tiêu đậm đà đưa cơm chuẩn vị mẹ nấu
   Score: 0.6749
   Type: Món kho
   Ingredients: ['300 gr Thịt heo quay', '1 củ Hành tím (cắt lát mỏng)', '3 muỗng canh Nước mắm', '1 ít Gia vị thông...
   Source: dienmayxanh

3. Thịt kho tộ thơm ngon đậm vị ăn cực bắt cơm
   Score: 0.6631
   Type: Món kho
   Ingredients: ['200 gr Thịt heo', '2 muỗng canh Nước mắm', '1 muỗng canh Dầu ăn', '1 muỗng canh Dầu điều', '1 ít G...
   Source: dienmayxanh

4. Hướng dẫn cách làm 2 món vịt nướng lá sen lá chuối ngon hấp dẫn
   Score: 0.6558
   Type: Món nướng
   Ingredients: ['1 con Vịt xiêm tơ', '1 ít Lá sen', '1 ít Rau r